# NB2A - Pokemon Data Processing
This notebook 

In [1]:
import json
import pandas as pd
import numpy as np

import sqlalchemy
from sqlalchemy import create_engine, text

In [2]:
with open('../../data/pokemon_data/generation_pokemon/generation_pokemon.json', 'r') as file:
    pokemon_data = json.load(file)

poke_df = pd.DataFrame(pokemon_data)
display(poke_df)

,name,url,generation,pokemon_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1,1
1,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/,1,2
2,venusaur,https://pokeapi.co/api/v2/pokemon-species/3/,1,3
3,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1,4
4,charmeleon,https://pokeapi.co/api/v2/pokemon-species/5/,1,5
...,...,...,...,...
1020,raging-bolt,https://pokeapi.co/api/v2/pokemon-species/1021/,9,1021
1021,iron-boulder,https://pokeapi.co/api/v2/pokemon-species/1022/,9,1022
1022,iron-crown,https://pokeapi.co/api/v2/pokemon-species/1023/,9,1023
1023,terapagos,https://pokeapi.co/api/v2/pokemon-species/1024/,9,1024


In [3]:
with open('../../data/pokemon_data/species_details/pokemon_1_species_details.json', 'r') as file:
    test_data = json.load(file)

test_df_1 = pd.json_normalize(test_data)
test_df_1.set_index('id', inplace = True)
display(test_df_1)
print(test_df_1.columns)

,base_happiness,capture_rate,egg_groups,evolves_from_species,form_descriptions,forms_switchable,gender_rate,genera,has_gender_differences,hatch_counter,...,flavor_text_entries.version.name,flavor_text_entries.version.url,generation.name,generation.url,growth_rate.name,growth_rate.url,habitat.name,habitat.url,shape.name,shape.url
id,,,,,,,,,,,,,,,,,,,,,
1,50,45,"[{'name': 'monster', 'url': 'https://pokeapi.c...",None,[],False,1,"[{'genus': 'たねポケモン', 'language': {'name': 'ja-...",False,20,...,red,https://pokeapi.co/api/v2/version/1/,generation-i,https://pokeapi.co/api/v2/generation/1/,medium-slow,https://pokeapi.co/api/v2/growth-rate/4/,grassland,https://pokeapi.co/api/v2/pokemon-habitat/3/,quadruped,https://pokeapi.co/api/v2/pokemon-shape/8/


Index(['base_happiness', 'capture_rate', 'egg_groups', 'evolves_from_species',
       'form_descriptions', 'forms_switchable', 'gender_rate', 'genera',
       'has_gender_differences', 'hatch_counter', 'is_baby', 'is_legendary',
       'is_mythical', 'name', 'names', 'order', 'pal_park_encounters',
       'pokedex_numbers', 'varieties', 'color.name', 'color.url',
       'evolution_chain.url', 'flavor_text_entries.flavor_text',
       'flavor_text_entries.language.name', 'flavor_text_entries.language.url',
       'flavor_text_entries.version.name', 'flavor_text_entries.version.url',
       'generation.name', 'generation.url', 'growth_rate.name',
       'growth_rate.url', 'habitat.name', 'habitat.url', 'shape.name',
       'shape.url'],
      dtype='object')


In [4]:
with open('../../data/pokemon_data/pokemon_details/pokemon_1_details.json', 'r') as file:
    test_data = json.load(file)

test_df_2 = pd.json_normalize(test_data)
test_df_2.set_index('id', inplace = True)
display(test_df_2)
print(test_df_2.columns)

,abilities,base_experience,forms,game_indices,height,held_items,is_default,location_area_encounters,moves,name,...,past_abilities,past_types,stats,types,weight,pokemon_portrait,cries.latest,cries.legacy,species.name,species.url
id,,,,,,,,,,,,,,,,,,,,,
1,"[{'ability': {'name': 'overgrow', 'url': 'http...",64,"[{'name': 'bulbasaur', 'url': 'https://pokeapi...","[{'game_index': 153, 'version': {'name': 'red'...",7,[],True,https://pokeapi.co/api/v2/pokemon/1/encounters,"[{'move': {'name': 'razor-wind', 'url': 'https...",bulbasaur,...,[],[],"[{'base_stat': 45, 'effort': 0, 'stat': {'name...","[{'slot': 1, 'type': {'name': 'grass', 'url': ...",69,https://raw.githubusercontent.com/PokeAPI/spri...,https://raw.githubusercontent.com/PokeAPI/crie...,https://raw.githubusercontent.com/PokeAPI/crie...,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/


Index(['abilities', 'base_experience', 'forms', 'game_indices', 'height',
       'held_items', 'is_default', 'location_area_encounters', 'moves', 'name',
       'order', 'past_abilities', 'past_types', 'stats', 'types', 'weight',
       'pokemon_portrait', 'cries.latest', 'cries.legacy', 'species.name',
       'species.url'],
      dtype='object')


In [5]:
print(test_df_2.iloc[0,-7])
print(test_df_2.iloc[0,-6])

[{'slot': 1, 'type': {'name': 'grass', 'url': 'https://pokeapi.co/api/v2/type/12/'}}, {'slot': 2, 'type': {'name': 'poison', 'url': 'https://pokeapi.co/api/v2/type/4/'}}]
69


In [6]:
range_of_poke_id = range(poke_df['pokemon_id'].min(), poke_df['pokemon_id'].max()+1)

In [7]:
def extract_pokemon_species_data(): 
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/species_details/pokemon_{pokemon_id}_species_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    
    poke_df['habitat_name'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['habitat']['name'] if json_data[x]['habitat'] else None)
    poke_df['pokemon_description'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['flavor_text_entries']['flavor_text'])
    return
    
extract_pokemon_species_data()

In [8]:
poke_df['pokemon_description'] = poke_df['pokemon_description'].str.replace('\n', ' ', regex=False)

In [9]:
def extract_pokemon_details_data():
    json_data = {
    pokemon_id: json.load(open(f'../../data/pokemon_data/pokemon_details/pokemon_{pokemon_id}_details.json', 'r'))
    for pokemon_id in range_of_poke_id
    }
    poke_df['type_1'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][0]['type']['name'])
    poke_df['type_2'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['types'][1]['type']['name'] if len(json_data[x]['types']) > 1 else None)
    poke_df['pokemon_portrait'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['pokemon_portrait'])

    stats = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']
    for i, stat in enumerate(stats):
        poke_df[f'{stat}_stat'] = poke_df['pokemon_id'].apply(lambda x: json_data[x]['stats'][i]['base_stat'])
        
    return 

extract_pokemon_details_data()

In [10]:
def calculate_total_stat(df: pd.DataFrame):
    stat_columns = [col for col in df.columns if 'stat' in col]
    df['total_stat'] = df[stat_columns].sum(axis=1)

calculate_total_stat(poke_df)

In [11]:
poke_df.set_index('pokemon_id', inplace = True)
fire_poke_df = poke_df[poke_df['type_1'] == 'fire']
ice_poke_df = poke_df[poke_df['type_1'] == 'ice']
water_poke_df = poke_df[poke_df['type_1'] == 'water']

In [12]:
poke_df['total_stat'].value_counts()
poke_df['type_1'].value_counts()

type_1
water       134
normal      118
grass       103
bug          83
fire         66
psychic      60
electric     59
rock         58
dark         45
poison       42
fighting     40
ground       40
dragon       37
steel        36
ghost        35
ice          31
fairy        29
flying        9
Name: count, dtype: int64

In [13]:
poke_df.to_json('../../data/pokemon_data/main_pokemon_df.json')

In [14]:
def assign_locations(df):
    df = df.copy()
    df.sort_values(by=['total_stat', 'hp_stat'], ascending=[False, False], inplace=True)
    df.loc[:,'ranking'] = range(1, len(df) + 1)
    return df

In [15]:
fire_poke_df = assign_locations(fire_poke_df)
display(fire_poke_df)

,name,url,generation,habitat_name,pokemon_description,type_1,type_2,pokemon_portrait,hp_stat,attack_stat,defense_stat,special_attack_stat,special_defense_stat,speed_stat,total_stat,ranking
pokemon_id,,,,,,,,,,,,,,,,
250,ho-oh,https://pokeapi.co/api/v2/pokemon-species/250/,2,rare,Legends claim this POKéMON flies the world's s...,fire,flying,https://raw.githubusercontent.com/PokeAPI/spri...,106,130,90,110,154,90,680,1
485,heatran,https://pokeapi.co/api/v2/pokemon-species/485/,4,None,It dwells in volcanic caves. It digs in with i...,fire,steel,https://raw.githubusercontent.com/PokeAPI/spri...,91,90,106,130,106,77,600,2
721,volcanion,https://pokeapi.co/api/v2/pokemon-species/721/,6,None,It lets out billows of steam and disappears in...,fire,water,https://raw.githubusercontent.com/PokeAPI/spri...,80,110,120,130,90,70,600,3
1020,gouging-fire,https://pokeapi.co/api/v2/pokemon-species/1020/,9,None,There are scant few reports of this creature b...,fire,dragon,https://raw.githubusercontent.com/PokeAPI/spri...,105,115,121,65,93,91,590,4
244,entei,https://pokeapi.co/api/v2/pokemon-species/244/,2,grassland,Volcanoes erupt when it barks. Un­ able to res...,fire,None,https://raw.githubusercontent.com/PokeAPI/spri...,115,115,85,90,75,100,580,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
322,numel,https://pokeapi.co/api/v2/pokemon-species/322/,3,mountain,NUMEL is extremely dull witted - it doesn’t no...,fire,ground,https://raw.githubusercontent.com/PokeAPI/spri...,60,60,40,65,45,35,305,62
850,sizzlipede,https://pokeapi.co/api/v2/pokemon-species/850/,8,None,It stores flammable gas in its body and uses i...,fire,bug,https://raw.githubusercontent.com/PokeAPI/spri...,50,65,45,50,50,45,305,63
37,vulpix,https://pokeapi.co/api/v2/pokemon-species/37/,1,grassland,"At the time of birth, it has just one tail. Th...",fire,None,https://raw.githubusercontent.com/PokeAPI/spri...,38,41,40,50,65,65,299,64


In [16]:
ice_poke_df = assign_locations(ice_poke_df)
display(ice_poke_df)

,name,url,generation,habitat_name,pokemon_description,type_1,type_2,pokemon_portrait,hp_stat,attack_stat,defense_stat,special_attack_stat,special_defense_stat,speed_stat,total_stat,ranking
pokemon_id,,,,,,,,,,,,,,,,
896,glastrier,https://pokeapi.co/api/v2/pokemon-species/896/,8,None,Glastrier emits intense cold from its hooves. ...,ice,None,https://raw.githubusercontent.com/PokeAPI/spri...,100,145,130,65,110,30,580,1
144,articuno,https://pokeapi.co/api/v2/pokemon-species/144/,1,rare,A legendary bird POKéMON that is said to appea...,ice,flying,https://raw.githubusercontent.com/PokeAPI/spri...,90,85,100,95,125,85,580,2
378,regice,https://pokeapi.co/api/v2/pokemon-species/378/,3,cave,REGICE’s body was made during an ice age. The ...,ice,None,https://raw.githubusercontent.com/PokeAPI/spri...,80,50,100,100,200,50,580,3
991,iron-bundle,https://pokeapi.co/api/v2/pokemon-species/991/,9,None,Its shape is similar to a robot featured in a ...,ice,water,https://raw.githubusercontent.com/PokeAPI/spri...,56,80,114,124,60,136,570,4
584,vanilluxe,https://pokeapi.co/api/v2/pokemon-species/584/,5,None,"Swallowing large amounts of water, they make s...",ice,None,https://raw.githubusercontent.com/PokeAPI/spri...,71,95,85,110,95,79,535,5
365,walrein,https://pokeapi.co/api/v2/pokemon-species/365/,3,sea,It swims through icy seas while shattering ice...,ice,water,https://raw.githubusercontent.com/PokeAPI/spri...,110,80,90,95,90,65,530,6
473,mamoswine,https://pokeapi.co/api/v2/pokemon-species/473/,4,None,Its impressive tusks are made of ice. The popu...,ice,ground,https://raw.githubusercontent.com/PokeAPI/spri...,110,130,80,70,60,80,530,7
471,glaceon,https://pokeapi.co/api/v2/pokemon-species/471/,4,None,"As a protective technique, it can completely f...",ice,None,https://raw.githubusercontent.com/PokeAPI/spri...,65,60,110,130,95,65,525,8
975,cetitan,https://pokeapi.co/api/v2/pokemon-species/975/,9,None,"This Pokémon wanders around snowy, icy areas. ...",ice,None,https://raw.githubusercontent.com/PokeAPI/spri...,170,113,65,45,55,73,521,9


In [17]:
water_poke_df = assign_locations(water_poke_df)

In [18]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS fire_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS fire_pokemon;'))
    conn.execute(create_tracks_statement)

fire_poke_df.to_sql("fire_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM fire_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 66


In [19]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS ice_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),
    generation VARCHAR(1),
    habitat_name VARCHAR(20), 
    pokemon_portrait VARCHAR(50),      
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS ice_pokemon;'))
    conn.execute(create_tracks_statement)

ice_poke_df.to_sql("ice_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM ice_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 31


In [20]:
database_path = "../../data/main.db"
engine = create_engine(f"sqlite:///{database_path}")
create_tracks_statement = text("""
CREATE TABLE IF NOT EXISTS water_pokemon (
    pokemon_id SMALLINT,
    name VARCHAR(20),
    url VARCHAR(50),  
    generation VARCHAR(1),
    habitat_name VARCHAR(20),
    pokemon_portrait VARCHAR(50),                                  
    pokemon_description TEXT,
    type_1 VARCHAR(20),
    type_2 VARCHAR(20),
    hp_stat SMALLINT,
    attack_stat SMALLINT,
    defense_stat SMALLINT,
    special_attack_stat SMALLINT,
    special_defense_stat SMALLINT,
    speed_stat SMALLINT,
    total_stat SMALLINT, 
    ranking SMALLINT PRIMARY KEY
);
""")

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS water_pokemon;'))
    conn.execute(create_tracks_statement)

water_poke_df.to_sql("water_pokemon", engine, if_exists = "append", index = True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM water_pokemon"))
    print(f"Number of rows in database: {result.scalar()}")

Number of rows in database: 134
